In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import joblib

best_iso_model = joblib.load("best_isolation_forest_model.pkl")
scaler = joblib.load("standard_scaler.pkl")


In [ ]:
evaluation_df=pd.read_csv("C:/EcoWatt-AI/data/processed/final_energy_anomaly_results.csv")

In [ ]:
feature_columns = [

    # Original electrical variables
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    # Temporal
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "year_sin",
    "year_cos",

    # Lag
    "active_power_lag_1",
    "active_power_lag_5",
    "active_power_lag_15",
    "active_power_lag_60",

    # Rolling
    "active_power_rolling_mean_15",
    "active_power_rolling_std_15",
    "active_power_rolling_min_15",
    "active_power_rolling_max_15",

    "active_power_rolling_mean_60",
    "active_power_rolling_std_60",
    "active_power_rolling_min_60",
    "active_power_rolling_max_60",

    # Behaviour
    "active_power_change_1",
    "deviation_from_15min_mean",
    "deviation_from_60min_mean",

    # Log transformed
    "active_power_change_rate_log",
    "rolling_zscore_15_log",
    "rolling_zscore_60_log"
]

In [ ]:
X = evaluation_df[feature_columns]
X_scaled = scaler.transform(X)

In [ ]:
print(shap.__version__)

In [ ]:
# Background data for SHAP
background = shap.sample(X_scaled, 100, random_state=42)

# Data to explain
explain_data = X_scaled[:100]

In [ ]:
explainer = shap.KernelExplainer(
    best_iso_model.decision_function,
    background
)

In [ ]:
shap_values = explainer.shap_values(explain_data)

In [ ]:
anomaly_index = evaluation_df[
    evaluation_df["predicted_anomaly"] == 1
].index[0]

print("Anomaly Index:", anomaly_index)

In [ ]:
shap.force_plot(
    explainer.expected_value,
    shap_values[anomaly_index % 100],
    explain_data[anomaly_index % 100],
    feature_names=feature_columns,
    matplotlib=True
)

In [ ]:
anomaly_indices = evaluation_df[
    evaluation_df["predicted_anomaly"] == 1
].index

print("Number of anomalies:", len(anomaly_indices))

sample_index = anomaly_indices[0]

print("Selected anomaly index:", sample_index)

In [ ]:
sample_position = sample_index % 100

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[sample_position],
        base_values=explainer.expected_value,
        data=explain_data[sample_position],
        feature_names=feature_columns
    )
)

In [ ]:
plt.figure(figsize=(12, 8))

shap.summary_plot(
    shap_values,
    explain_data,
    feature_names=feature_columns,
    show=False
)

plt.tight_layout()
plt.savefig("C:/EcoWatt-AI/results/shap_summary_plot.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

shap.summary_plot(
    shap_values,
    explain_data,
    feature_names=feature_columns,
    plot_type="bar",
    show=False
)

plt.tight_layout()
plt.savefig("C:/EcoWatt-AI/results/shap_feature_importance.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Mean |SHAP| Value": np.abs(shap_values).mean(axis=0)
})

feature_importance = feature_importance.sort_values(
    "Mean |SHAP| Value",
    ascending=False
)

feature_importance.head(20)

In [ ]:
top10_features = feature_importance.head(10)

print(top10_features)

In [ ]:
report_table = feature_importance.head(10).copy()

report_table.columns = [
    "Feature",
    "Importance Score"
]

report_table

In [ ]:
report_table.to_csv(
    "C:/EcoWatt-AI/data/processed/top10_feature_importance.csv",
    index=False
)

In [ ]:
import os

files = [
    "best_isolation_forest_model.pkl",
    "standard_scaler.pkl",
    "final_energy_anomaly_results.csv",
    "top_20_energy_anomalies.csv",
    "dataset_summary.csv",
    "shap_summary_plot.png",
    "shap_feature_importance.png",
    "shap_feature_importance.csv",
    "top10_feature_importance.csv"
]

for file in files:
    if os.path.exists(file):
        print(f"✅ {file}")
    else:
        print(f"❌ Missing: {file}")